# Outlier Detection

**Topic:** Exploratory Data Analysis

In [7]:
import numpy as np
import pandas as pd
import seaborn as sns

import plotly.graph_objects as go
import plotly.express as px

import ipywidgets as widgets
from ipywidgets import Dropdown, FloatSlider, Output, HBox, VBox

from IPython.display import display, clear_output
from scipy import stats

titanic = sns.load_dataset("titanic")
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this session you will be able to:

- **Apply** the IQR method and the z-score method to flag potential outliers in a numeric column
- **Interpret** what the IQR multiplier controls and how changing it shifts the boundary between normal and extreme
- **Evaluate** whether a flagged outlier is a data error or a genuine extreme value

> **Tip:** Start with the IQR method and drag the multiplier slider from 1.5 down to 0.5. Watch how the number of flagged outliers grows as you tighten the threshold. Then switch to Z-score and compare which fares each method flags.

---
## How we got here

In `05_handling_duplicates.ipynb` we checked for exact duplicate records. Now we look at a different kind of data quality problem: values that are present and unique, but extreme. Outlier detection draws on the dispersion concepts from `statistics/05_dispersion.ipynb` and the standardization techniques from `statistics/15_zscores_standardization.ipynb`. This is where that statistical foundation meets real data.

---
## Why this matters for data science

Outliers affect almost every statistical calculation and many machine learning algorithms. A single extreme fare value in the Titanic dataset pulls the mean upward and inflates the standard deviation. Tree-based models can handle outliers relatively well, but linear models, distance-based models, and neural networks are all sensitive to extreme values in the input features. Detecting outliers early lets you make an informed decision before they quietly corrupt your analysis.

---
## Try it yourself

In [8]:
fare = titanic["fare"].dropna()

method_dropdown = Dropdown(
    options=["IQR method", "Z-score method"],
    value="IQR method",
    description="Method:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="300px"),
)

iqr_slider = FloatSlider(
    value=1.5, min=0.5, max=4.0, step=0.25,
    description="IQR multiplier:",
    style={"description_width": "130px"},
    layout=widgets.Layout(width="420px"),
)

_layout = base_layout(title="", xaxis_title="Fare (£)", yaxis_title="")
_layout.update(showlegend=True, height=320)

fw = go.FigureWidget(
    data=[
        go.Box(
            name="Normal range",
            x=[0],
            y=["Normal"],
            orientation="h",
            marker_color=PALETTE["primary"],
            boxpoints=False,
            hoverinfo="skip",
        ),
        go.Scatter(
            mode="markers",
            marker=dict(opacity=0, size=14),
            showlegend=False,
            hovertemplate="<b>%{customdata}</b>: %{x:.1f}<extra></extra>",
        ),
        go.Scatter(
            mode="markers",
            name="Flagged outliers",
            marker=dict(color=PALETTE["secondary"], size=8, symbol="x"),
        ),
    ],
    layout=_layout,
)

def render(change=None):
    method = method_dropdown.value
    k = iqr_slider.value

    Q1 = fare.quantile(0.25)
    Q3 = fare.quantile(0.75)
    IQR = Q3 - Q1
    iqr_lower = Q1 - k * IQR
    iqr_upper = Q3 + k * IQR
    z_scores = np.abs(stats.zscore(fare))

    if method == "IQR method":
        is_outlier = (fare < iqr_lower) | (fare > iqr_upper)
        threshold_label = f"IQR bounds: [{iqr_lower:.1f}, {iqr_upper:.1f}]  (k={k})"
    elif method == "Z-score method":
        is_outlier = z_scores > 3
        threshold_label = "Z-score threshold: |z| > 3"


    n_outliers = is_outlier.sum()
    normal_vals = fare[~is_outlier]
    outlier_vals = fare[is_outlier]

    bq1 = normal_vals.quantile(0.25)
    bq3 = normal_vals.quantile(0.75)
    biqr = bq3 - bq1
    bmedian = normal_vals.median()
    lower_whisker = normal_vals[normal_vals >= bq1 - 1.5 * biqr].min()
    upper_whisker = normal_vals[normal_vals <= bq3 + 1.5 * biqr].max()

    with fw.batch_update():
        fw.data[0].x = normal_vals
        fw.data[0].y = ["Normal"] * len(normal_vals)
        fw.data[1].x = [lower_whisker, bq1, bmedian, bq3, upper_whisker]
        fw.data[1].y = ["Normal"] * 5
        fw.data[1].customdata = ["lower bound", "Q1", "median", "Q3", "upper bound"]
        fw.data[2].x = outlier_vals.values
        fw.data[2].y = ["Normal"] * n_outliers
        fw.data[2].name = f"Flagged outliers ({n_outliers})"
        fw.layout.title.text = f"Titanic Fare Distribution — {threshold_label}"

method_dropdown.observe(render, names="value")
iqr_slider.observe(render, names="value")

display(VBox([HBox([method_dropdown]), iqr_slider, fw]))
render()

---
## What's happening?

Two methods dominate outlier detection in EDA:

**IQR method** (from `statistics/05_dispersion.ipynb`):

```python
Q1 = df['fare'].quantile(0.25)
Q3 = df['fare'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
outliers = df[(df['fare'] < lower) | (df['fare'] > upper)]
```

The multiplier `k=1.5` is the conventional default. Increasing it to 3.0 flags only the most extreme values; decreasing it to 0.5 flags many more.

**Z-score method** (from `statistics/15_zscores_standardization.ipynb`):

```python
from scipy import stats
z_scores = np.abs(stats.zscore(df['fare']))
outliers = df[z_scores > 3]
```

A z-score above 3 means the value is more than 3 standard deviations from the mean. The two methods often agree, but the IQR method is more robust when the distribution is skewed.

---
## Real-world example: Titanic Dataset

The chart below shows the fare distribution using the IQR method with k=1.5. Points beyond the upper bound are flagged in orange.

Notice:
- The **highest fare paid was £512**, far above the IQR upper bound of around £65
- Several passengers paid over £200, which is unusual but historically plausible for first-class berths
- The box plot reveals how compressed the majority of fares are below £50, while a long right tail extends to extreme values

> **Discussion question:** The most expensive fare was £512. Is that a data entry error or a genuine price paid by a wealthy first-class passenger? How would you decide?

In [9]:
np.random.seed(42)

fare = titanic["fare"].dropna()
Q1 = fare.quantile(0.25)
Q3 = fare.quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

is_outlier = fare > upper_bound
normal_fare = fare[~is_outlier]
outlier_fare = fare[is_outlier]

fig = go.Figure()

fig.add_trace(go.Box(
    x=normal_fare,
    name="Normal range",
    marker_color=PALETTE["primary"],
    boxpoints=False,
))

fig.add_trace(go.Scatter(
    x=outlier_fare,
    y=[0] * len(outlier_fare),
    mode="markers",
    name=f"IQR outliers ({is_outlier.sum()} passengers)",
    marker=dict(color=PALETTE["secondary"], size=9, symbol="x"),
))

fig.add_vline(
    x=upper_bound, line_dash="dash",
    line_color=PALETTE["accent"],
    annotation_text=f"IQR upper bound: £{upper_bound:.1f}",
    annotation_position="top right",
)

layout = base_layout(
    title="Titanic Fare Distribution with IQR Outlier Bounds (k=1.5)",
    xaxis_title="Fare (£)",
    yaxis_title="",
)
layout.update(showlegend=True, height=340)
fig.update_layout(layout)
fig.show()

### Outlier handling strategies across real-world scenarios

| Context | Likely cause | Recommended action |
|---------|-------------|-------------------|
| Salary data with one entry of $0 | Data entry error | Investigate, then replace or drop |
| House price with one extreme mansion | Genuine extreme value | Keep, but consider log-transforming the column |
| Age recorded as 999 | Sentinel / placeholder value | Replace with NaN and treat as missing |
| Titanic fare of £512 | Genuine first-class berth price | Keep — it is historically consistent |
| Duplicate transaction inflating total | Duplicate + outlier combined | Dedup first, then re-examine |

---
## Key takeaway

> **Flagging an outlier is the easy part. Deciding whether it is an error to remove or a real signal to keep requires domain knowledge, and that judgment belongs to you, not an algorithm.**

---
*Next up: 07_descriptive_statistics_in_practice.ipynb — applying all of our statistical tools to the Titanic data for the first time*